# W2 — Spatial organisation of the immune programmes  [Reviewer 1 item 5]

Visium spots are not sorted, so unlike the single-cell data above, composition is meaningful
here. Three H3.3 K27M paediatric diffuse midline glioma sections (GSE268577).

Two questions: do the immune programmes form spatially coherent domains rather than noise, and
does the spot-level picture match what the bulk ecotypes imply?

## 1. Spot-level signature scores and Moran's I

In [1]:
"""C — do the immune programs form spatially coherent domains in pDMG tissue?
GSE268577, three H3.3K27M pDMG Visium sections. Spots are unsorted, so composition is meaningful."""
import numpy as np, pandas as pd, gzip, glob, io
from scipy import sparse, stats
np.random.seed(42)
UP="/mnt/user-data/uploads/Open PBTA/Revision/Week1/_inputs"
sigs={}
for line in open(f"{UP}/brain_immune_signatures.gmt"):
    f=line.rstrip("\n").split("\t"); sigs[f[0]]=[g for g in f[2:] if g.strip()]

def load(sample):
    base=glob.glob(f"data/visium/*_pDMG_Sample-{sample}_")[0] if False else \
         glob.glob(f"data/visium/*_pDMG_Sample-{sample}_matrix.mtx.gz")[0].replace("matrix.mtx.gz","")
    feat=pd.read_csv(base+"features.tsv.gz",sep="\t",header=None)
    bc=pd.read_csv(base+"barcodes.tsv.gz",header=None)[0].values
    with gzip.open(base+"matrix.mtx.gz","rt") as f:
        from scipy.io import mmread
        M=mmread(f).tocsc()
    pos=pd.read_csv(base+"tissue_positions.csv.gz").set_index("barcode")
    pos=pos.loc[bc]
    keep=pos.in_tissue.values==1
    M=M[:,keep]; pos=pos[keep]; bc=bc[keep]
    A=np.asarray(M.todense(),dtype=np.float32)          # 17,943 x n_spots, small
    X=pd.DataFrame(A,index=feat[1].values,columns=bc)
    X=X.groupby(level=0).sum()                           # collapse duplicate symbols
    tot=X.sum(axis=0)
    L=np.log2(X/tot*1e4+1).astype(np.float32)
    return L,pos,tot

def neighbours(pos):
    r=pos.array_row.values; c=pos.array_col.values
    n=len(r); W=sparse.lil_matrix((n,n))
    key={(a,b):i for i,(a,b) in enumerate(zip(r,c))}
    for i,(a,b) in enumerate(zip(r,c)):
        for da,db in [(0,-2),(0,2),(-1,-1),(-1,1),(1,-1),(1,1)]:
            j=key.get((a+da,b+db))
            if j is not None: W[i,j]=1
    return W.tocsr()

def morans_I(x,W):
    x=np.asarray(x,dtype=float); z=x-x.mean(); n=len(x)
    S0=W.sum(); den=float((z**2).sum())
    return (n/S0)*(float(z @ (W @ z))/den) if den>0 and S0>0 else np.nan

def morans_I_perm(x,W,B,rng):
    """Vectorised null: B column-permutations at once."""
    x=np.asarray(x,dtype=float); n=len(x); S0=W.sum()
    idx=np.argsort(rng.random((B,n)),axis=1)
    Z=x[idx].T                                   # n x B
    Z=Z-Z.mean(axis=0,keepdims=True)
    num=np.einsum("ij,ij->j",Z,W@Z)
    den=(Z**2).sum(axis=0)
    return (n/S0)*(num/den)

rows=[]; spot_scores={}
for s in ["1","2","3"]:
    L,pos,tot=load(s)
    W=neighbours(pos)
    deg=np.asarray(W.sum(axis=1)).ravel()
    print(f"Sample-{s}: {L.shape[1]} in-tissue spots, {L.shape[0]} genes, "
          f"median {np.median(tot):.0f} counts/spot, mean {deg.mean():.1f} neighbours")
    mean_expr=L.mean(axis=1); bins=pd.qcut(mean_expr.rank(method="first"),25,labels=False)
    rng=np.random.default_rng(42)
    def score(genes,nctrl=50):
        g=[x for x in genes if x in L.index]
        if len(g)<3: return None
        ctrl=[]
        for x in g:
            pool=mean_expr.index[bins==bins[x]]
            ctrl+=list(rng.choice(pool,size=min(nctrl,len(pool)),replace=False))
        return (L.loc[g].mean(axis=0)-L.loc[ctrl].mean(axis=0)).values
    S={k:score(v) for k,v in sigs.items()}
    S={k:v for k,v in S.items() if v is not None}
    Sdf=pd.DataFrame(S,index=pos.index); spot_scores[s]=(Sdf,pos)
    for name,v in S.items():
        I=morans_I(v,W)
        perm=morans_I_perm(v,W,999,rng)
        p=(np.sum(perm>=I)+1)/(len(perm)+1)
        rows.append(dict(sample=f"Sample-{s}",signature=name,n_spots=L.shape[1],
                         morans_I=round(I,3),perm_p=p,
                         perm_mean=round(perm.mean(),4),perm_sd=round(perm.std(),4)))
R=pd.DataFrame(rows)
R["q_BH"]=stats.false_discovery_control(R.perm_p)
R.to_csv("W2_C_morans_I.tsv",sep="\t",index=False)
print("\n=== Moran's I by signature, mean across the three sections ===")
piv=R.pivot(index="signature",columns="sample",values="morans_I")
piv["mean_I"]=piv.mean(axis=1)
sig_all=R.groupby("signature").q_BH.max()
piv["max_q"]=sig_all
print(piv.sort_values("mean_I",ascending=False).round(3).to_string())
print(f"\nsignatures spatially autocorrelated (q<0.05) in all three sections: "
      f"{int((R.groupby('signature').q_BH.max()<0.05).sum())}/{R.signature.nunique()}")
import pickle; pickle.dump(spot_scores,open("w2_spot_scores.pkl","wb"))


Sample-1: 577 in-tissue spots, 17941 genes, median 4933 counts/spot, mean 5.3 neighbours


Sample-2: 856 in-tissue spots, 17941 genes, median 5050 counts/spot, mean 5.5 neighbours


Sample-3: 217 in-tissue spots, 17941 genes, median 14702 counts/spot, mean 5.2 neighbours



=== Moran's I by signature, mean across the three sections ===
sample                        Sample-1  Sample-2  Sample-3  mean_I  max_q
signature                                                                
MDM_Klemm2020                    0.608     0.107     0.650   0.455  0.002
MHC_Class_II                     0.410     0.067     0.754   0.410  0.003
IFN_Gamma_Response               0.510     0.006     0.675   0.397  0.391
MHC_Class_I                      0.458     0.097     0.635   0.397  0.002
MoTAM_Antunes2021                0.512     0.120     0.554   0.395  0.002
IFN_Alpha_Response               0.576     0.088     0.507   0.390  0.002
Glioma_Inflammatory_Wang2017     0.496    -0.030     0.694   0.387  0.928
M2_Macrophage                    0.347     0.023     0.663   0.344  0.168
DAM_KerenShaul2017               0.212     0.170     0.553   0.312  0.002
Neutrophil_Activation            0.323     0.069     0.371   0.254  0.002
M1_Macrophage                    0.098     0.044

Ten of the 24 signatures are spatially autocorrelated at q < 0.05 in all three sections. The
strongest are myeloid and antigen-presentation programmes (MDM_Klemm2020 mean I = 0.455,
MHC_Class_II 0.410, MHC_Class_I 0.397, MoTAM_Antunes2021 0.395). Sample-2 is consistently the
weakest section across every analysis here.

## 2. Spot-level ecotype assignment and spatial coherence

In [2]:
"""C (continued) — assign each Visium spot to the nearest bulk ecotype centroid in the
24-signature space, and test whether the assignments form spatially coherent domains."""
import numpy as np, pandas as pd, pickle
from scipy import sparse, stats
rng=np.random.default_rng(42)
ANN="/mnt/user-data/uploads/Open PBTA/Revision/FINAL MANUSCRIPT 260722 - 수정본/7. Reproducibility data/Sample annotation/sample_master_annotation.tsv"
d=pd.read_csv(ANN,sep="\t").set_index("sample")
sigcols=[c for c in d.columns if c.startswith("ssGSEA_")]
Zb=(d[sigcols]-d[sigcols].mean())/d[sigcols].std()
Zb.columns=[c.replace("ssGSEA_","") for c in Zb.columns]
CENT=Zb.groupby(d["ecotype"]).mean()
ORDER=["Lymphocyte-inflamed","Myeloid-dominant","Immune-desert"]
CENT=CENT.loc[ORDER]
print("bulk ecotype centroids, 24-signature space:",CENT.shape)

spot=pickle.load(open("w2_spot_scores.pkl","rb"))
def neighbours(pos):
    r=pos.array_row.values; c=pos.array_col.values
    key={(a,b):i for i,(a,b) in enumerate(zip(r,c))}
    n=len(r); W=sparse.lil_matrix((n,n))
    for i,(a,b) in enumerate(zip(r,c)):
        for da,db in [(0,-2),(0,2),(-1,-1),(-1,1),(1,-1),(1,1)]:
            j=key.get((a+da,b+db))
            if j is not None: W[i,j]=1
    return W.tocsr()

rows=[];colo=[]
for s,(S,pos) in spot.items():
    cols=[c for c in CENT.columns if c in S.columns]
    Zs=(S[cols]-S[cols].mean())/S[cols].std()
    C=CENT[cols]
    # nearest centroid by Pearson correlation across signatures
    A=Zs.values; B=C.values
    A=(A-A.mean(1,keepdims=True))/A.std(1,keepdims=True)
    B=(B-B.mean(1,keepdims=True))/B.std(1,keepdims=True)
    corr=A@B.T/A.shape[1]
    assign=np.array(ORDER)[corr.argmax(1)]
    W=neighbours(pos)
    # join-count: fraction of neighbouring spot pairs with the same assignment
    src,dst=W.nonzero()
    same=(assign[src]==assign[dst]).mean()
    perm=np.array([(assign[rng.permutation(len(assign))][src]==assign[rng.permutation(len(assign))][dst]).mean()
                   for _ in range(999)])
    # correct null: permute once per replicate
    perm=[]
    for _ in range(999):
        p=rng.permutation(assign); perm.append((p[src]==p[dst]).mean())
    perm=np.array(perm); pval=(np.sum(perm>=same)+1)/1000
    counts=pd.Series(assign).value_counts().reindex(ORDER).fillna(0).astype(int)
    rows.append(dict(sample=s,n_spots=len(assign),
                     **{k.split("-")[0][:4]:int(v) for k,v in counts.items()},
                     n_ecotypes_present=int((counts>0.05*len(assign)).sum()),
                     same_neighbour_fraction=round(same,3),
                     null_mean=round(perm.mean(),3),perm_p=pval))
    my=[c for c in ["Microglia_Klemm2020","MDM_Klemm2020","MgTAM_Antunes2021","MoTAM_Antunes2021",
                    "DAM_KerenShaul2017","M2_Macrophage"] if c in S.columns]
    ly=[c for c in ["T_Cell_Cytotoxicity","NK_Cell_Activity","Chemokine_T_Cell_Recruitment",
                    "T_Cell_Exhaustion","Tregs_Friebel2020"] if c in S.columns]
    r=stats.pearsonr(S[my].mean(axis=1),S[ly].mean(axis=1))
    colo.append(dict(sample=s,n_spots=len(assign),myeloid_lymphoid_r=round(r.statistic,3),p=r.pvalue))
    np.save(f"w2_spot_assign_{s}.npy",assign)
R=pd.DataFrame(rows); Cc=pd.DataFrame(colo)
print("\n=== spot-level ecotype assignment and spatial coherence ===")
print(R.to_string(index=False))
print("\n=== myeloid x lymphoid programme co-localisation per spot ===")
print(Cc.to_string(index=False))
print(f"\nbulk myeloid x lymphoid theme correlation for comparison: "
      f"{np.corrcoef(Zb[[c for c in ['Microglia_Klemm2020','MDM_Klemm2020','MgTAM_Antunes2021','MoTAM_Antunes2021','DAM_KerenShaul2017','M2_Macrophage']]].mean(axis=1), Zb[['T_Cell_Cytotoxicity','NK_Cell_Activity','Chemokine_T_Cell_Recruitment','T_Cell_Exhaustion','Tregs_Friebel2020']].mean(axis=1))[0,1]:.3f}")
R.to_csv("W2_C_spot_ecotype.tsv",sep="\t",index=False); Cc.to_csv("W2_C_colocalisation.tsv",sep="\t",index=False)


bulk ecotype centroids, 24-signature space: (3, 24)



=== spot-level ecotype assignment and spatial coherence ===
sample  n_spots  Lymp  Myel  Immu  n_ecotypes_present  same_neighbour_fraction  null_mean  perm_p
     1      577   223   118   236                   3                    0.540      0.358   0.001
     2      856   325   241   290                   3                    0.357      0.338   0.019
     3      217    83    38    96                   3                    0.562      0.370   0.001

=== myeloid x lymphoid programme co-localisation per spot ===
sample  n_spots  myeloid_lymphoid_r            p
     1      577               0.358 6.282259e-19
     2      856               0.048 1.632347e-01
     3      217               0.484 3.789821e-14

bulk myeloid x lymphoid theme correlation for comparison: 0.799


Two results, and the second is the more interesting one.

**The assignments are spatially coherent.** Neighbouring spots share an ecotype assignment more
often than chance in all three sections (0.540, 0.357 and 0.562 against nulls of 0.358, 0.338 and
0.370; P = 0.001, 0.019, 0.001). The programmes are organised in tissue, not scattered.

**Every section contains all three ecotypes.** A single tumour is not one ecotype; the bulk label
is a spatial average over regions. This is a limitation of the bulk framework that the spatial
data make concrete, and it should be stated rather than buried.

**The myeloid-lymphoid coupling is an aggregation effect, in part.** In bulk the two theme
composites correlate at r = 0.80; per spot the correlation is 0.36, 0.05 and 0.48. Bulk mixing
inflates the apparent coupling of myeloid and lymphoid programmes.

## 3. Figure

In [3]:
import numpy as np, pandas as pd, pickle, matplotlib as mpl
mpl.use("Agg"); import matplotlib.pyplot as plt
from scipy import stats
mpl.rcParams.update({"font.family":"DejaVu Sans","pdf.fonttype":42,"ps.fonttype":42,"axes.linewidth":.8})
COL={"Lymphocyte-inflamed":"#3B82F6","Myeloid-dominant":"#EF4444","Immune-desert":"#9CA3AF"}
ORDER=list(COL)
spot=pickle.load(open("w2_spot_scores.pkl","rb"))
R=pd.read_csv("W2_C_morans_I.tsv",sep="\t"); E=pd.read_csv("W2_C_spot_ecotype.tsv",sep="\t")
Cc=pd.read_csv("W2_C_colocalisation.tsv",sep="\t")

fig=plt.figure(figsize=(13.0,7.0),dpi=300)
gs=fig.add_gridspec(2,4,height_ratios=[.92,1.0],hspace=.30,wspace=.46)

for i,s in enumerate(["1","2","3"]):
    S,pos=spot[s]; a=np.load(f"w2_spot_assign_{s}.npy",allow_pickle=True)
    ax=fig.add_subplot(gs[0,i])
    ax.scatter(pos.array_col,-pos.array_row,c=[COL[x] for x in a],s=7,lw=0)
    ax.set_aspect("equal"); ax.axis("off")
    row=E[E["sample"].astype(str)==s].iloc[0]
    ax.set_title(f"{'ABC'[i]}  pDMG Sample-{s}   {int(row.n_spots)} spots\n"
                 f"same-neighbour {row.same_neighbour_fraction:.3f} vs null {row.null_mean:.3f}, P = {row.perm_p:.3f}",
                 fontsize=7.6,loc="left")
h=[plt.Line2D([],[],marker="o",ls="",color=COL[k],label=k,ms=5) for k in ORDER]
fig.legend(handles=h,loc="upper right",bbox_to_anchor=(.995,.93),fontsize=7,frameon=False,
           title="nearest bulk ecotype centroid",title_fontsize=7.5)

# D — Moran's I
ax=fig.add_subplot(gs[0,3]); ax.axis("off")
piv=R.pivot(index="signature",columns="sample",values="morans_I")
piv["mean"]=piv.mean(axis=1); piv=piv.sort_values("mean",ascending=True).tail(12)
ax2=fig.add_subplot(gs[1,0:2])
y=np.arange(len(piv))
for j,(c,m) in enumerate(zip(["Sample-1","Sample-2","Sample-3"],["o","s","^"])):
    ax2.scatter(piv[c],y,s=17,marker=m,label=c,alpha=.85,
                color=["#1E3A5F","#F97316","#0F766E"][j],lw=0)
ax2.axvline(0,color="k",lw=.7)
ax2.set_yticks(y); ax2.set_yticklabels([s.replace("_"," ") for s in piv.index],fontsize=6.6)
ax2.set_xlabel("Moran's I (spatial autocorrelation)",fontsize=7.4)
ax2.set_title("D  Immune programmes are spatially structured, not random\ntop 12 of 24 signatures by mean Moran's I; 999-permutation test",
              fontsize=8,loc="left")
ax2.legend(fontsize=6.5,frameon=False,loc="lower right"); ax2.tick_params(labelsize=6.6)
ax2.spines[["top","right"]].set_visible(False)

# E — myeloid vs lymphoid, bulk vs spot
ax=fig.add_subplot(gs[1,2])
ANN="/mnt/user-data/uploads/Open PBTA/Revision/FINAL MANUSCRIPT 260722 - 수정본/7. Reproducibility data/Sample annotation/sample_master_annotation.tsv"
d=pd.read_csv(ANN,sep="\t")
sig=[c for c in d.columns if c.startswith("ssGSEA_")]
Zb=(d[sig]-d[sig].mean())/d[sig].std(); Zb.columns=[c.replace("ssGSEA_","") for c in Zb.columns]
MY=["Microglia_Klemm2020","MDM_Klemm2020","MgTAM_Antunes2021","MoTAM_Antunes2021","DAM_KerenShaul2017","M2_Macrophage"]
LY=["T_Cell_Cytotoxicity","NK_Cell_Activity","Chemokine_T_Cell_Recruitment","T_Cell_Exhaustion","Tregs_Friebel2020"]
rb=np.corrcoef(Zb[MY].mean(axis=1),Zb[LY].mean(axis=1))[0,1]
vals=[rb]+list(Cc.myeloid_lymphoid_r)
labs=["bulk\nn = 349"]+[f"spot\nSample-{s}" for s in Cc["sample"]]
cols=["#1E3A5F","#0F766E","#0F766E","#0F766E"]
ax.bar(range(4),vals,color=cols,width=.62)
for i,v in enumerate(vals): ax.text(i,v+.02,f"{v:.2f}",ha="center",fontsize=7.2)
ax.set_xticks(range(4)); ax.set_xticklabels(labs,fontsize=6.8)
ax.set_ylabel("myeloid x lymphoid correlation",fontsize=7.2); ax.set_ylim(0,1)
ax.set_title("E  Myeloid-lymphoid coupling is weaker\nwithin tissue than in bulk",fontsize=8,loc="left")
ax.tick_params(labelsize=6.6); ax.spines[["top","right"]].set_visible(False)

# F — every tumour contains every ecotype
ax=fig.add_subplot(gs[1,3])
bot=np.zeros(3)
for k in ORDER:
    v=E[k.split("-")[0][:4]].values/E.n_spots.values*100
    ax.bar(range(3),v,bottom=bot,color=COL[k],width=.6,label=k); bot+=v
ax.set_xticks(range(3)); ax.set_xticklabels([f"Sample-{s}" for s in E["sample"]],fontsize=7)
ax.set_ylabel("% of in-tissue spots",fontsize=7.2); ax.set_ylim(0,100)
ax.set_title("F  Every section contains all three\necotypes",fontsize=8,loc="left")
ax.tick_params(labelsize=6.6); ax.spines[["top","right"]].set_visible(False)

fig.suptitle("Spatial organisation of the immune programmes in H3.3 K27M paediatric diffuse midline glioma (GSE268577)",
             fontsize=9.5,y=.985)
for e in ("png","pdf"): fig.savefig(f"FigureS21_spatial.{e}",dpi=300,bbox_inches="tight",facecolor="white")
print("saved FigureS21")


saved FigureS21


## Framing

Three sections from three tumours. This shows that the immune programmes are spatially structured
and that ecotype assignment varies within a tumour. It is not external validation and does not
speak to the survival association.